# Quickstart: The Core Pipeline
CodeGraphene bridges the gap between static analysis and Large Language Models (LLMs).
It does this in three steps:
1. **Parse**: Convert raw source code into a mathematical Code Property Graph (CPG).
2. **Trim**: Extract only the subgraph that is relevant to a specific target.
3. **Serialize**: Flatten that subgraph back into a clean text prompt for an LLM.

## Setup and Imports
We will be analyzing `sample_code.py`, which contains a snippet from the RepoCoder `function_level_completion_4k_context_codex` dataset.

In [1]:
from codegraphene.core import NodeGranularity
from codegraphene.parsers.joern import JoernParser
from codegraphene.trimmers.khop import KHopTrimmer
from codegraphene.serializers.text import CodeReconstructionSerializer
from codegraphene.pipeline import GraphPipeline

target_file = "sample_code.py"

## Configuring the Pipeline
In keeping with the three steps mentioned above, we build our GraphPipeline by defining a parser, a trimmer, and a serializer. 

In [2]:
# Parser: JoernParser, which uses Joern under the hood.
parser = JoernParser(granularity=NodeGranularity.LINE)

# Trimmer: A simple K-hop trimmer. The number of hops can be configured. Here, we use hops=1 for illustrative purposes.
trimmer = KHopTrimmer(hops=1)

# Serializer: CodeReconstructionSerializer, which turns our trimmer subgraph back into code.
serializer = CodeReconstructionSerializer(granularity=NodeGranularity.LINE)

pipeline = GraphPipeline(
    parser=parser,
    trimmer=trimmer,
    serializer=serializer
)

## Running the pipeline
In this example, we defined our pipeline components to function at line-level granularity (see `01_granularity.ipynb` for more information). We define a target line and run the pipeline on our file while targeting that line.

In [3]:
target_line = 63
result = pipeline.run(file_path=target_file, target=target_line)

# run() always returns a PipelineResult, regardless of whether the
# pipeline completed fully or short-circuited on a raw parser export.
print("\n--- FINAL PROMPT ---")
print(result.output)

[Pipeline] Step 1: running JoernParser on sample_code.py...
[JoernParser] Parsing source code at: sample_code.py
[JoernParser] Running: joern-parse sample_code.py --output /tmp/tmpso_waucf/cpg.bin


[JoernParser] Running: joern-export /tmp/tmpso_waucf/cpg.bin --repr all --out /tmp/tmpso_waucf/export


[JoernParser] Ingesting DOT file into NetworkX...


[Pipeline] Target resolved to node(s) ['25769803796', '30064771187', '30064771188', '30064771189', '30064771190', '55834574893', '55834574894', '68719476837', '68719476838', '68719476839', '68719476840', '94489280568'].
[Pipeline] Step 2: running KHopTrimmer...
[Pipeline] 12 targets matched; running KHopTrimmer once per target and unioning results.


[Pipeline] Step 3: running CodeReconstructionSerializer...

--- FINAL PROMPT ---
Line 52: self
Line 59: self.cur_batch
Line 62: tmp7 = self.scheduler
self.scheduler.step()
Line 63: tmp7 = self.scheduler
self.scheduler.step()
Line 66: tmp9
Line 67: self
Line 71: self
